# Phase 2 — Downstream Fine-Tuning & Experiment Grid

**Goal:** Load pre-trained SSL checkpoints and run the full (method × strategy × label_fraction × seed) grid.

**Expects checkpoints from `01_pretrain_ssl.ipynb` in `/content/checkpoints/`:**
- `simclr_encoder_tailored.pth`
- `mae_encoder.pth`
- `mae_improved_encoder.pth`

**Output:** `results/results_main.csv` (4 methods × 3 strategies × 3 label fractions × 3 seeds = 108 rows).

---
**Runtime:** ~15–30 min per seed × 3 seeds on Colab T4. Use `seeds=[42]` for a quick single-seed run.

## 1 — Setup

In [ ]:
# Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!git clone https://github.com/Friedrich-233/ST456_GroupProject.git /content/ST456_GroupProject
!pip install timm -q
import sys
sys.path.insert(0, '/content/ST456_GroupProject')

In [ ]:
from pathlib import Path

DRIVE_DATA_DIR  = Path('/content/drive/MyDrive/ST456 Group project/pcamv1')
DATA_DIR         = Path('/content/pcam_data')
CHECKPOINT_DIR   = Path('/content/ST456_GroupProject/checkpoints')
RESULTS_DIR      = Path('/content/ST456_GroupProject/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Results:     {RESULTS_DIR}")

## 2 — Load Data

In [ ]:
from data import load_all_data, build_downstream_loaders, LABEL_FRACTIONS

print("Loading PCam data...")
data = load_all_data(
    drive_data_dir=DRIVE_DATA_DIR,
    data_dir=DATA_DIR,
    pretrain_fraction=0.15,
    downstream_pool_fraction=0.15,
    use_subset=True,
)

train_loaders, val_loader, test_loader = build_downstream_loaders(data)
print("\nLabel fractions:", list(LABEL_FRACTIONS.keys()))
print("Train loaders:", {k: len(v.dataset) for k, v in train_loaders.items()})

## 3 — Verify Checkpoints

In [ ]:
import os

required_ckpts = {
    'simclr':    CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    'mae':       CHECKPOINT_DIR / 'mae_encoder.pth',
    'mae_improved': CHECKPOINT_DIR / 'mae_improved_encoder.pth',
}

print("Checkpoint status:")
all_ok = True
for name, path in required_ckpts.items():
    exists = path.exists()
    status = '✓' if exists else '✗ MISSING'
    if not exists:
        all_ok = False
    size = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else ''
    print(f"  {status}  {name:<14}  {size}")

if not all_ok:
    raise FileNotFoundError(
        'One or more checkpoints are missing. '
        'Run 01_pretrain_ssl.ipynb first to generate them.'
    )
print('\nAll checkpoints found. Ready to run experiments.')

## 4 — Quick Single-Run Test (one config only)

Run a single (method, strategy, label) to verify everything works before launching the full grid.

In [ ]:
from training import run_single_experiment, TrainConfig

# Single test run
test_result = run_single_experiment(
    method='mae_improved',
    strategy='frozen',
    label_name='1%',
    train_loader=train_loaders['1%'],
    val_loader=val_loader,
    test_loader=test_loader,
    mae_improved_checkpoint=CHECKPOINT_DIR / 'mae_improved_encoder.pth',
    seed=42,
)
print(f"\nTest run — MAE Improved, frozen, 1%:")
print(f"  test_auc = {test_result['test_auc']:.4f}")
print(f"  test_acc = {test_result['test_accuracy']:.4f}")
print(f"  test_f1  = {test_result['test_f1']:.4f}")

## 5 — Full Experiment Grid

> ⚠️ **This takes ~6–12 h on Colab T4.**
>
> **Quick sanity check:** comment out this cell and use `seeds=[42]` below.
> **Full grid:** `seeds=[42, 52, 62]`.

Methods:
- `supervised_from_scratch` — random init ResNet-18 baseline
- `simclr` — SimCLR with tailored augmentations
- `mae` — base MAE (patch=8, depth=6)
- `mae_improved` — improved MAE (patch=4, depth=12, cosine LR, EMA)

Strategies:
- `frozen` — encoder frozen, only linear head trained
- `partial` — last encoder stage / last transformer blocks unfrozen
- `full` — full fine-tune

In [ ]:
# ═══════════════════════════════════════════════════════
# Grid A — Seed 42
# ═══════════════════════════════════════════════════════
results_42 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[42],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    mae_improved_checkpoint=CHECKPOINT_DIR / 'mae_improved_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_42.to_csv(RESULTS_DIR / 'results_seed42.csv', index=False)
print(f"Seed 42 done — {len(results_42)} rows → {RESULTS_DIR / 'results_seed42.csv'}")
display(results_42)


In [ ]:
# ═══════════════════════════════════════════════════════
# Grid B — Seed 52
# ═══════════════════════════════════════════════════════
results_52 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[52],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    mae_improved_checkpoint=CHECKPOINT_DIR / 'mae_improved_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_52.to_csv(RESULTS_DIR / 'results_seed52.csv', index=False)
print(f"Seed 52 done — {len(results_52)} rows → {RESULTS_DIR / 'results_seed52.csv'}")
display(results_52)


In [ ]:
# ═══════════════════════════════════════════════════════
# Grid C — Seed 62
# ═══════════════════════════════════════════════════════
results_62 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[62],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    mae_improved_checkpoint=CHECKPOINT_DIR / 'mae_improved_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_62.to_csv(RESULTS_DIR / 'results_seed62.csv', index=False)
print(f"Seed 62 done — {len(results_62)} rows → {RESULTS_DIR / 'results_seed62.csv'}")
display(results_62)


## 6 — Merge Results & Final Preview\n

In [ ]:
# ═══════════════════════════════════════════════════════
# Merge all seeds and save final results
# ═══════════════════════════════════════════════════════
import pandas as pd

results_df = pd.concat([results_42, results_52, results_62], ignore_index=True)
results_df.to_csv(RESULTS_DIR / 'results_main.csv', index=False)
print(f"Combined: {len(results_df)} rows (3 seeds × 4 methods × 3 strategies × 3 label fractions)")
print(f"Saved → {RESULTS_DIR / 'results_main.csv'}")
display(results_df)


## 7 — Optional: SimCLR Generic vs Tailored Ablation

In [ ]:
# Run this only if you trained a 'generic' SimCLR checkpoint as well.
# Point simclr_generic_checkpoint to wherever you saved it.

simclr_generic_checkpoint = CHECKPOINT_DIR / 'simclr_encoder_generic.pth'

if simclr_generic_checkpoint.exists():
    results_simclr_generic = run_experiment_grid(
        methods=['simclr'],
        strategies=FINETUNE_STRATEGIES,
        label_names=list(LABEL_FRACTIONS.keys()),
        seeds=[42, 52, 62],
        train_loaders=train_loaders,
        val_loader=val_loader,
        test_loader=test_loader,
        simclr_checkpoint=simclr_generic_checkpoint,
        mae_checkpoint=None,
        mae_improved_checkpoint=None,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )
    results_simclr_generic.to_csv(RESULTS_DIR / 'results_simclr_generic.csv', index=False)
    print(f"SimCLR generic ablation saved → {RESULTS_DIR / 'results_simclr_generic.csv'}")
else:
    print('Generic SimCLR checkpoint not found — skipping ablation.')

## 8 — Results Preview (mean ± std across seeds)

In [ ]:
from evaluation import summarise_results, build_report_table

summary = summarise_results(results_df)
report_table = build_report_table(summary)
display(report_table)

print("\nNext step: open 03_analysis.ipynb for visualizations and the report table.")